## Middleware

Middleware provides a way to more tightly control what happens inside an agent. Middleware is useful for the following:
- Tracking agent behaviour with logging, analytics and debugging
- Transforming prompts, tool selection and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate-limits, guardrails and Pll detection

In [1]:
import os
from dotenv import load_dotenv
import anthropic
from langchain.tools import tool
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")

model = init_chat_model("claude-sonnet-5")

### Summarization Middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for:
- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters

In [2]:
agent_with_Middleware = create_agent(
  model = "claude-sonnet-5",
  checkpointer = InMemorySaver(),
  middleware=[
    SummarizationMiddleware(
      model = "claude-haiku-4-5",
      trigger = ("messages", 10),
      keep = ("messages", 4)
    )
  ]
)

In [3]:
## Run with thread id
config = {"configurable" : {"thread_id" : "test-1"}}

In [4]:
## Test data for run

questions = [
  "What is 2+2",
  "What is 10*15",
  "What is 20/5",
  "what is 5*4",
  "what is 3*3",
  "what is 4*4"
]

In [5]:
for q in questions:
  response = agent_with_Middleware.invoke({"messages" : [HumanMessage(content=q)]}, config)
  print(f"Messages: {response}")
  print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2', additional_kwargs={}, response_metadata={}, id='201ebb12-e156-490b-9ca0-9912da1dbb73'), AIMessage(content="# 2 + 2 = 4\n\nThis is a basic arithmetic addition problem. Is there anything else you'd like to calculate or any other questions you have?", additional_kwargs={}, response_metadata={'id': 'msg_011Cg7283kd46r57HTyM3oRf', 'container': None, 'model': 'claude-sonnet-5', 'stop_details': None, 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 12, 'output_tokens': 51, 'output_tokens_details': {'thinking_tokens': 0}, 'server_tool_use': None, 'service_tier': 'standard', 'speed': 'standard'}, 'diagnostics': None, 'model_name': 'claude-sonnet-5', 'model_provider': 'anthropic'}, id='lc_run--01a147b0-2502-78f1-b5c5-06deff68b7ec-0', tool

## Summarization using token size

In [6]:

@tool
def search_hotels(city : str) -> str:
  """Search Hotels - returns long response to use more tokens"""
  return f"""Hotels in {city}:
  1. Grand Hotel - 5 stars, $300/night, spa, pool, gym
  2. City Inn - 4 star, $180/night, business center
  3. Budget Stay - 3 star, $75/night, free wifi"""


agent_with_token_middleware = create_agent(
  model="claude-haiku-4-5",
  tools=[search_hotels],
  checkpointer=InMemorySaver(),
  middleware=[
    SummarizationMiddleware(
        model="claude-haiku-4-5",
        trigger= ("tokens", 500),
        keep = ("tokens", 200)
      )
  ]
)

# token counter - approx
def count_tokens(messages):
  total_chars = sum(len(str(m.content)) for m in messages)
  return total_chars //4 # 4 chars = 1 token (assuming)

In [8]:
# run test

cities = ["Paris", "London", "Tokyo", "New York","Dubai", "Singapore"]

for city in cities:
  response = agent_with_token_middleware.invoke(
    {"messages": [
      HumanMessage(content=f"Find hotels in {city}")
    ]},
    config
  )

  tokens = count_tokens(response["messages"])
  print(f"{city}: - {tokens}")
  print(f"{response['messages']}")



Paris: - 626
[HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nFind and compare hotels across multiple cities (Paris, London, Tokyo, New York, Dubai, and Singapore), providing user with options and detailed information to support booking decisions.\n\n## SUMMARY\n\nUser requested hotel searches in six cities: Paris, London, Tokyo, New York, Dubai, and Singapore. All searches returned identical results across all locations:\n- Grand Hotel (5-star, $300/night): spa, pool, gym\n- City Inn (4-star, $180/night): business center\n- Budget Stay (3-star, $75/night): free wifi\n\nAll three hotels presented for each city. User has not indicated preference for specific hotel, city, or provided filtering criteria (price range, amenities, star rating). No booking decisions made. Notable pattern: identical hotel inventory and pricing across all cities suggests data limitation or uniform search results.\n\n## ARTIFACTS\n\nNone\n\n## NEXT STEPS\n\nAwait user

## Fraction

In [9]:
agent_with_tokenFraction_middleware = create_agent(
  model="claude-haiku-4-5",
  tools=[search_hotels],
  checkpointer=InMemorySaver(),
  middleware=[
    SummarizationMiddleware(
        model="claude-haiku-4-5",
        trigger= ("fraction", 0.005),
        keep = ("fraction", 0.002)
      )
  ]
)

for city in cities:
  response = agent_with_tokenFraction_middleware.invoke(
    {"messages": [
      HumanMessage(content=f"Find hotels in {city}")
    ]},
    config
  )

  tokens = count_tokens(response["messages"])
  print(f"{city}: - {tokens}")
  print(f"{response['messages']}")

Paris: - 189
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='217b2d6d-0b83-43b0-aea2-cd3660390a16'), AIMessage(content=[{'id': 'toolu_01VG2L8NaAN4ebKcFatLQSP7', 'caller': {'type': 'direct'}, 'input': {'city': 'Paris'}, 'name': 'search_hotels', 'type': 'tool_use', 'toolset_name': None}], additional_kwargs={}, response_metadata={'id': 'msg_011Cg7C6F9bBzg8cSimc5rmQ', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 565, 'output_tokens': 54, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard', 'speed': 'standard'}, 'diagnostics': None, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'}, id='lc_run--01a14827-d

## Human in the loop

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. 

Human-in-the-loop is useful for the following:
- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id : str) -> str:
  """Mock function to read email by its id"""
  return f"Email content for the id : {email_id}"

def send_email_tool(recipient: str, subject: str, body:str) -> str:
  """Mock function to send an email"""
  return f"Email sent to {recipient} with the subject '{subject}'"

In [11]:
agent_with_humaninloop_middleware01 = create_agent(
  model="claude-haiku-4-5",
  tools=[read_email_tool, send_email_tool],
  checkpointer=InMemorySaver(),
  middleware=[
    HumanInTheLoopMiddleware(
      interrupt_on={
        "send_email_tool":{
          "allowed_decisions":["approve", "edit", "reject"]          
        },
        "read_email_tool" : False
      }
    )
  ]
)

In [14]:
result = agent_with_humaninloop_middleware01.invoke({
  "messages": [HumanMessage(
    content="Send email to john@email.com with subject 'Hello' and body 'How are you?'"
  )],
},
config=config)

In [ ]:
result # check the result we have something called __interrupt__

{'messages': [HumanMessage(content="Send email to john@email.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='27388f54-13fb-4bac-afb0-8e3bc94d0eee'),
  AIMessage(content=[{'id': 'toolu_017rvxGngFL5tVcuaC2pBhLt', 'caller': {'type': 'direct'}, 'input': {'recipient': 'john@email.com', 'subject': 'Hello', 'body': 'How are you?'}, 'name': 'send_email_tool', 'type': 'tool_use', 'toolset_name': None}], additional_kwargs={}, response_metadata={'id': 'msg_011Cg7DgSLrMK5hn8piBGaNZ', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 670, 'output_tokens': 97, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard', 'speed': 'standard'}, 'di

In [18]:
from langgraph.types import Command

# approving
if "__interrupt__" in result:
  print("Paused !! Waiting for approval")

  result = agent_with_humaninloop_middleware01.invoke(
    Command(resume={
      "decisions":[{"type" : "approve"}]
    }),
    config=config
  )

  print(f"Result : {result['messages'][-1].content}")

In [19]:
# Rejecting

if "__interrupt__" in result:
  print("Paused !! Waiting for approval")

  result = agent_with_humaninloop_middleware01.invoke(
    Command(resume={
      "decisions":[{"type" : "reject"}]
    }),
    config=config
  )

  print(f"Result : {result['messages'][-1].content}")